# 🌫️ Karachi Air Quality & Weather — EDA
**Data:** 90 days of hourly data from Open-Meteo  
**City:** Karachi, Pakistan (24.86°N, 67.01°E)

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats

sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'figure.figsize': (14, 4)})
pd.set_option('display.float_format', '{:.2f}'.format)
print('✅ Libraries loaded')

## 1 · Load Data

In [ ]:
df = pd.read_csv('karachi_weather_90d.csv', index_col='time', parse_dates=True)
print(f'Shape : {df.shape}')
print(f'Range : {df.index.min()}  →  {df.index.max()}')
df.head(3)

## 2 · Data Quality Check

In [ ]:
print('── dtypes ──')
print(df.dtypes)
print('\n── missing values ──')
print(df.isnull().sum())

In [ ]:
df.describe().T.style.background_gradient(cmap='Blues', subset=['mean','std','50%'])

## 3 · AQI Time Series

In [ ]:
fig, ax = plt.subplots(figsize=(15, 4))
ax.plot(df.index, df['us_aqi'], lw=0.8, color='steelblue', alpha=0.7, label='Hourly AQI')
ax.plot(df['us_aqi'].resample('D').mean(), lw=2, color='navy', label='Daily mean')

bands = [(0,50,'#00e400','Good'), (50,100,'#ffff00','Moderate'),
         (100,150,'#ff7e00','USG'), (150,200,'#ff0000','Unhealthy'),
         (200,300,'#8f3f97','Very Unhealthy')]
for lo, hi, c, lbl in bands:
    ax.axhspan(lo, hi, alpha=0.07, color=c, label=lbl)

ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=2))
plt.xticks(rotation=30)
ax.set_title('Karachi — Hourly US AQI (90 days)', fontsize=14)
ax.set_ylabel('AQI')
ax.legend(loc='upper right', fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

## 4 · Pollutant Distributions

In [ ]:
pollutants = ['pm2_5','pm10','ozone','nitrogen_dioxide','sulphur_dioxide','carbon_monoxide']
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, col in zip(axes.flat, pollutants):
    data = df[col].dropna()
    ax.hist(data, bins=40, color='steelblue', edgecolor='white', alpha=0.85)
    ax.axvline(data.mean(),   color='red',    lw=1.5, linestyle='--', label=f'mean={data.mean():.1f}')
    ax.axvline(data.median(), color='orange', lw=1.5, linestyle='--', label=f'median={data.median():.1f}')
    ax.set_title(col); ax.set_xlabel('µg/m³ or ppb')
    ax.legend(fontsize=7)
plt.suptitle('Karachi — Pollutant Distributions', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 5 · Temporal Patterns

In [ ]:
df['hour']        = df.index.hour
df['day_of_week'] = df.index.day_name()
df['month']       = df.index.month_name()
df['date']        = df.index.date

fig, axes = plt.subplots(1, 2, figsize=(16, 4))

# by hour
hourly = df.groupby('hour')['us_aqi'].mean()
axes[0].bar(hourly.index, hourly.values, color='coral', edgecolor='white')
axes[0].set_title('Average AQI by Hour of Day')
axes[0].set_xlabel('Hour (PKT)'); axes[0].set_ylabel('Mean AQI')
axes[0].set_xticks(range(0,24,2))

# by day of week
dow_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
dow = df.groupby('day_of_week')['us_aqi'].mean().reindex(dow_order)
axes[1].bar(dow.index, dow.values, color='mediumpurple', edgecolor='white')
axes[1].set_title('Average AQI by Day of Week')
axes[1].set_xlabel('Day'); axes[1].set_ylabel('Mean AQI')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap — hour vs day-of-week
pivot = df.pivot_table(values='us_aqi', index='hour', columns='day_of_week',
                        aggfunc='mean')[dow_order]
fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(pivot, cmap='YlOrRd', annot=False, linewidths=0.3, ax=ax,
            cbar_kws={'label': 'Mean AQI'})
ax.set_title('AQI Heatmap — Hour of Day × Day of Week', fontsize=13)
ax.set_ylabel('Hour (PKT)')
plt.tight_layout()
plt.show()

## 6 · Weather vs AQI Relationships

In [ ]:
weather_vars = ['temperature_2m','relative_humidity_2m','wind_speed_10m',
                'precipitation','surface_pressure','cloud_cover']
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, var in zip(axes.flat, weather_vars):
    ax.scatter(df[var], df['us_aqi'], alpha=0.15, s=4, color='steelblue')
    # regression line
    mask = df[[var,'us_aqi']].dropna()
    m, b, r, p, _ = stats.linregress(mask[var], mask['us_aqi'])
    xs = np.linspace(mask[var].min(), mask[var].max(), 100)
    ax.plot(xs, m*xs+b, color='red', lw=1.5)
    ax.set_xlabel(var); ax.set_ylabel('US AQI')
    ax.set_title(f'{var}  (r={r:.2f}, p={p:.3f})')
plt.suptitle('Weather Variables vs AQI', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 7 · Correlation Heatmap

In [ ]:
num_cols = ['us_aqi','pm2_5','pm10','ozone','nitrogen_dioxide',
            'sulphur_dioxide','carbon_monoxide',
            'temperature_2m','relative_humidity_2m',
            'wind_speed_10m','wind_direction_10m',
            'precipitation','surface_pressure','cloud_cover',
            'visibility','shortwave_radiation']
num_cols = [c for c in num_cols if c in df.columns]
corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(13, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))   # upper triangle
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            square=True, linewidths=0.4, ax=ax,
            cbar_kws={'shrink': 0.8}, annot_kws={'size': 7})
ax.set_title('Feature Correlation Matrix', fontsize=14)
plt.tight_layout()
plt.show()

## 8 · Outlier Detection (Box Plots)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# pollutants box plot
df[['pm2_5','pm10','ozone','nitrogen_dioxide','sulphur_dioxide']].plot.box(
    ax=axes[0], vert=False, patch_artist=True,
    boxprops=dict(facecolor='lightblue', color='navy'))
axes[0].set_title('Pollutant Spread (Box Plot)')

# weather box plot
df[['temperature_2m','relative_humidity_2m','wind_speed_10m','cloud_cover']].plot.box(
    ax=axes[1], vert=False, patch_artist=True,
    boxprops=dict(facecolor='lightyellow', color='darkorange'))
axes[1].set_title('Weather Spread (Box Plot)')
plt.tight_layout()
plt.show()

## 9 · Rolling Statistics

In [ ]:
fig, ax = plt.subplots(figsize=(15, 4))
ax.plot(df['us_aqi'],                        alpha=0.3, lw=0.7, color='steelblue', label='Hourly')
ax.plot(df['us_aqi'].rolling(24).mean(),     lw=1.8,   color='navy',   label='24h rolling mean')
ax.plot(df['us_aqi'].rolling(24*7).mean(),   lw=2,     color='red',    label='7-day rolling mean')
ax.fill_between(df.index,
                df['us_aqi'].rolling(24).mean() - df['us_aqi'].rolling(24).std(),
                df['us_aqi'].rolling(24).mean() + df['us_aqi'].rolling(24).std(),
                alpha=0.15, color='navy', label='±1 std (24h)')
ax.set_title('AQI Rolling Mean & Std Dev', fontsize=13)
ax.set_ylabel('AQI'); ax.legend(fontsize=9)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 10 · Daily AQI Category Breakdown

In [ ]:
def aqi_category(v):
    if v <= 50:   return 'Good'
    elif v <= 100: return 'Moderate'
    elif v <= 150: return 'USG'
    elif v <= 200: return 'Unhealthy'
    elif v <= 300: return 'Very Unhealthy'
    else:          return 'Hazardous'

df['aqi_category'] = df['us_aqi'].apply(aqi_category)
cat_order  = ['Good','Moderate','USG','Unhealthy','Very Unhealthy','Hazardous']
cat_colors = ['#00e400','#ffff00','#ff7e00','#ff0000','#8f3f97','#7e0023']
cat_counts = df['aqi_category'].value_counts().reindex(cat_order).dropna()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(cat_counts.index, cat_counts.values,
            color=[cat_colors[cat_order.index(c)] for c in cat_counts.index],
            edgecolor='white')
axes[0].set_title('Hours per AQI Category'); axes[0].set_ylabel('Hours')
plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=20)

axes[1].pie(cat_counts.values, labels=cat_counts.index,
            colors=[cat_colors[cat_order.index(c)] for c in cat_counts.index],
            autopct='%1.1f%%', startangle=140)
axes[1].set_title('AQI Category Share')
plt.tight_layout()
plt.show()

print('\n📊 Category Summary:')
print(cat_counts.to_frame('hours').assign(pct=lambda x: (x['hours']/len(df)*100).round(1)))

## 11 · Wind Direction vs AQI (Polar Plot)

In [ ]:
if 'wind_direction_10m' in df.columns:
    fig = plt.figure(figsize=(7, 7))
    ax  = fig.add_subplot(111, projection='polar')
    theta = np.deg2rad(df['wind_direction_10m'])
    sc = ax.scatter(theta, df['wind_speed_10m'],
                    c=df['us_aqi'], cmap='YlOrRd',
                    alpha=0.3, s=5)
    plt.colorbar(sc, ax=ax, label='US AQI', shrink=0.7)
    ax.set_theta_zero_location('N'); ax.set_theta_direction(-1)
    ax.set_title('Wind Direction & Speed\n(colour = AQI)', va='bottom', fontsize=12)
    plt.tight_layout()
    plt.show()

## 12 · Key Insights Summary

In [ ]:
worst_hour = df.groupby('hour')['us_aqi'].mean().idxmax()
best_hour  = df.groupby('hour')['us_aqi'].mean().idxmin()
worst_day  = df.groupby('day_of_week')['us_aqi'].mean().idxmax()
top_corr   = df[num_cols].corr()['us_aqi'].drop('us_aqi').abs().idxmax()

print('='*50)
print('KARACHI AQI — KEY EDA FINDINGS')
print('='*50)
print(f"Mean AQI          : {df['us_aqi'].mean():.1f}")
print(f"Max  AQI          : {df['us_aqi'].max():.0f}  on {df['us_aqi'].idxmax().date()}")
print(f"Min  AQI          : {df['us_aqi'].min():.0f}  on {df['us_aqi'].idxmin().date()}")
print(f"Worst hour of day : {worst_hour}:00")
print(f"Best  hour of day : {best_hour}:00")
print(f"Worst day of week : {worst_day}")
print(f"Strongest corr.   : {top_corr}  (r={df[num_cols].corr()['us_aqi'][top_corr]:.2f})")
print(f"% Unhealthy hours : {(df['us_aqi'] > 150).mean()*100:.1f}%")